# M3 CLV 조건부 카테고리 전이 실패 원인 진단 - Dunnhumby

이미 완료된 seed 42 실험(`fdf845b331c2`)의 **실제 CLV와 CLV-shuffle 체크포인트만 다시 읽어** 실패 경로를 분해합니다. 모델을 다시 학습하거나 checkpoint를 선택하지 않습니다. 이 수정본은 기존 프로젝트 모듈 캐시를 제거하고 진단 코드 `v2`를 확인한 뒤 실행합니다.

1. 실제 CLV와 shuffle이 고객별 보조 그래프를 얼마나 다르게 만들었는지 확인합니다.
2. 최종 점수를 `기준 LightGCN 점수 + CLV 조건부 전이 점수`로 나누어, 보조 점수가 후보와 정답에서 실제로 작동했는지 확인합니다.
3. @10·20·50에서 실제 CLV와 shuffle의 추천 집합이 얼마나 달라졌는지 전체 및 CLV 5분위별로 확인합니다.

DAY 1--683 학습과 DAY 684--690 탐색 평가를 그대로 재구성하며, 최종 test나 holdout은 만들지 않습니다. 이 결과는 다음 가설을 만드는 기술적 사후 진단일 뿐이며 유의성·일반화·모형 선택을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, shutil, subprocess, sys

REVIEWED_SHA = '09d16c623ac1dcd45422758ac7962d672f44fae4'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name == 'clv_run_state' or module_name.startswith(('lightgcn_clv', 'clv_m3')):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('Pinned diagnostic source:', actual_sha)
print('Removed cached project modules; the next cell will import v2 from this checkout.')

In [ ]:
import json, torch
import lightgcn_clv_m3_category_transition_diagnostic as diagnostic

assert diagnostic.CODE_VERSION == 'm3-clv-category-transition-failure-diagnostic-v2', diagnostic.CODE_VERSION
assert str(Path(diagnostic.__file__).resolve()).startswith(str(repo.resolve()) + '/'), diagnostic.__file__
cfg = diagnostic.configure_m3_category_transition_diagnostic()
assert torch.cuda.is_available(), 'Colab 런타임에서 GPU를 선택한 뒤 다시 실행하세요.'
summary = diagnostic.preflight_summary(cfg)
assert summary['code_version'] == 'm3-clv-category-transition-failure-diagnostic-v2'
assert summary['training'] is False
assert summary['checkpoint_selection'] is False
assert summary['final_test_constructed'] is False
assert summary['holdout_constructed'] is False
assert summary['rank_limit'] == 50
assert summary['source_result_id'] == 'fdf845b331c2'
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
graph_summary = diagnostic.run_m3_category_transition_diagnostic(cfg)

In [ ]:
from IPython.display import display

print('1. 실제 CLV와 shuffle의 보조 그래프 차이')
display(graph_summary)

print('2. 실제 CLV와 shuffle의 배정 차이')
display(graph_summary.attrs['clv_assignment_correlation'])

print('3. 기준 점수와 CLV 조건부 전이 점수의 크기')
display(graph_summary.attrs['score_components'])

print('4. 정답과 추천후보에 부여된 CLV 조건부 전이 점수 차이')
display(graph_summary.attrs['score_truth_candidate_contrast'])

print('5. @10·20·50 추천목록 변경 정도')
display(graph_summary.attrs['recommendation_overlap'])

print('6. 데이터·체크포인트·추천목록 품질검사')
display(graph_summary.attrs['quality_checks'])

print('기술적 판독:')
print(json.dumps(graph_summary.attrs['diagnostic_reading'], ensure_ascii=False, indent=2))
print('결과 파일:', graph_summary.attrs['result_paths'])